In [ ]:
# %pip install scipy

In [2]:
import numpy as np
from scipy import linalg
from scipy.special import gamma

# from math import gamma, pi, log

In [ ]:
# gamma
# we can use from math
# gamma(1)
# or we can use it from scipy.special.gamma

# negative infinity
# np.negative(np.inf)
# -np.inf
# `np.NINF` was removed in the NumPy 2.0 release. Use `-np.inf` instead.

# pi value
# from math import pi
# or access as np.pi

# log function
# log is also in math
# or we can use it from numpy as np.log

In [ ]:
# Input: data matrix {Y, U, V}, and
# hyper-parameters {tilde_lambda0, tilde_lambda1, alpha, beta ,
#                   lambda0, lambda1, alpha, beta,
#                   K, xi , eta}

# def sigmoid(x):
#     return 1 / (1 + np.exp(-x))


def group_lasso_density(vec, group_size, lambda_):
    norm = linalg.norm(vec)
    density = (
        2 ** (-group_size)
        * np.pi ** (-(group_size - 1) / 2)
        / gamma((group_size + 1) / 2)
        * np.exp(-norm * lambda_)
    )
    return density


def p_star(vec, theta, group_size, lambda0, lambda1):
    spike = (1 - theta) * group_lasso_density(vec, group_size, lambda0)
    slab = theta * group_lasso_density(vec, group_size, lambda1)
    p_star = slab / (spike + slab)
    return p_star


def lambda_star(p_star, lambda0, lambda1):
    lambda_star_value = (1 - p_star) * lambda0 + p_star * lambda1
    return lambda_star_value


def update_momentum(x, x_lag, iter):
    momentum = x + (iter - 2) / (iter + 1) * (x - x_lag)
    return momentum


def h_function(lambda_star, p_star, lambda1, eta):
    h_value = (lambda_star - lambda1) ** 2 + 2 / eta * p_star
    return h_value


def update_delta(h_value, eta, p_star, lambda0, lambda1):
    if h_value > 0:
        delta = np.sqrt(2 * eta * np.log(1 / p_star)) + eta * lambda1
    else:
        delta = eta * lambda_star(p_star, lambda0, lambda1)
    return delta


def update_count(mat):
    count = 0
    d = mat.shape[1]
    for i in range(d):
        if linalg.norm(mat[:, i], 0) != 0:
            count += 1
    return count, d


def update_theta(count, d, alpha, beta):
    theta = (alpha + count) / (alpha + beta + d)
    return theta

In [ ]:
def get_W(Y, mu, U, V, A, B, xi):
    W = (1 + xi * Y + Y) / (
        1 + np.exp(-np.outer(mu, np.ones(Y.shape[1])) - U @ A @ B.T @ V.T)
    )
    return W


def gradient(direction, mat, Y, U, V, A, B, xi):
    grad = direction * (W - mat)
    return grad

np.float64(2.302585092994046)

In [45]:
u = np.array([[1, 2, 3], [1, 2, 3]])
v = np.array([[1, 1, 1], [5, 5, 5], [6, 6, 6]])
rank1_matrix = np.outer(u, v)
rank1_matrix

array([[ 1,  1,  1,  5,  5,  5,  6,  6,  6],
       [ 2,  2,  2, 10, 10, 10, 12, 12, 12],
       [ 3,  3,  3, 15, 15, 15, 18, 18, 18],
       [ 1,  1,  1,  5,  5,  5,  6,  6,  6],
       [ 2,  2,  2, 10, 10, 10, 12, 12, 12],
       [ 3,  3,  3, 15, 15, 15, 18, 18, 18]])

In [ ]:
np.array([[1, 2, 3], [2, 4, 5]]).T

array([[1, 2],
       [2, 4],
       [3, 5]])

In [47]:
u.flatten()

array([1, 2, 3, 1, 2, 3])

In [ ]:
# define matrix completion problem
import numpy as np
from scipy.linalg import svd


def matrix_completion(M, mask, rank, n_iter=100, tol=1e-4):
    """
    Perform matrix completion using singular value thresholding.
    """
    # Initialize the completed matrix
    X = np.zeros_like(M)
    for i in range(n_iter):
        # Perform SVD
        U, s, Vt = svd(X, full_matrices=False)
        # Keep only the largest singular values
        s = np.maximum(s - tol, 0)
        # Reconstruct the matrix
        X = np.dot(U, np.dot(np.diag(s), Vt))
        # Apply the mask
        X *= mask
        # Check for convergence
        if np.linalg.norm(M - X) < tol:
            break
    return X

In [2]:
# define matrix completion problem
import numpy as np

In [ ]:
np.arange(3 * 4 * 5 * 6).reshape((3, 4, 5, 6))
# 3*4*5*6
a = np.arange(3 * 4 * 5 * 6).reshape((3, 4, 5, 6))
b = np.arange(3 * 4 * 5 * 6)[::-1].reshape((5, 4, 6, 3))
np.dot(a, b)[2, 3, 2, 1, 2, 2]
# 499128
# sum(a[2,3,2,:] * b[1,2,:,2])
# 499128

np.int64(499128)

In [ ]:
np.dot(a, b)[1, 1, 1, 1, 1]

array([248634, 247683, 246732])

In [14]:
help(np.dot)

Help on _ArrayFunctionDispatcher in module numpy:

dot(...)
    dot(a, b, out=None)

    Dot product of two arrays. Specifically,

    - If both `a` and `b` are 1-D arrays, it is inner product of vectors
      (without complex conjugation).

    - If both `a` and `b` are 2-D arrays, it is matrix multiplication,
      but using :func:`matmul` or ``a @ b`` is preferred.

    - If either `a` or `b` is 0-D (scalar), it is equivalent to
      :func:`multiply` and using ``numpy.multiply(a, b)`` or ``a * b`` is
      preferred.

    - If `a` is an N-D array and `b` is a 1-D array, it is a sum product over
      the last axis of `a` and `b`.

    - If `a` is an N-D array and `b` is an M-D array (where ``M>=2``), it is a
      sum product over the last axis of `a` and the second-to-last axis of
      `b`::

        dot(a, b)[i,j,k,m] = sum(a[i,j,:] * b[k,:,m])

    It uses an optimized BLAS library when possible (see `numpy.linalg`).

    Parameters
    ----------
    a : array_like
        Fir